# JobKB — inspection & QA

Spot-checks over the `kb/` outputs of `run_pipeline.py`. The KB is built by
**ingest → hierarchy → align → attach → merge → qa**; each stage can be re-run alone
(`python run_pipeline.py --stages merge`, `--from align`, `--add NAME`, ...).

Alignment uses `bge-m3` embeddings + `mDeBERTa-v3-base` NLI (no human review): a semantic
occupation merge needs strong embedding **and** mutual NLI entailment **and** the same ISCO
group; ISCO attachment is NLI-re-ranked. This notebook checks integrity, de-duplication, the
merge gate, attachment, and the neutral skill ontology.

In [1]:
import os, sys, pandas as pd
sys.path.insert(0, '..')
pd.set_option('display.max_colwidth', 80)
KB = os.path.join('..', 'kb')
def load(name):
    return pd.read_csv(os.path.join(KB, name), dtype=str, keep_default_na=False)
def n_members(series):  # member_entity_ids is ' | '-joined; split literally (not as regex)
    return series.apply(lambda s: len(s.split(' | ')) if s else 0)

occ   = load('occupations.csv')
skl   = load('skills.csv')
labels= load('labels.csv')
rels  = load('occupation_skill_relations.csv')
hier  = load('hierarchy.csv')
align = load('concept_alignments.csv')
uocc  = load('unified_occupations.csv')
uskl  = load('unified_skills.csv')
prov  = load('provenance.csv')
prov[['source', 'source_version', 'retrieval_method', 'notes']]

,source,source_version,retrieval_method,notes
0,ALIGNMENT,-,embed:st+nli:on,"46843 alignments, 106 merge-flagged"
1,ATTACH,-,embed:BAAI/bge-m3+nli:MoritzLaurer/mDeBERTa-v3-base-mnli-xnli,"140 attached, top-3 NLI re-ranked (54 NLI overrode embedding top-1; 23 low-c..."
2,ISCO,ISCO-08,official_en_csv,"23 groups, 20 edges (IT branches 25/35/133)"
3,ESCO,ESCO v1.2 (en),official_en_csv,"110 occ, 1828 skills, 6613 relations"
4,NOC,NOC 2021 v1.0,official_en_fr_csv,15 IT unit-group occupations (bilingual)
5,ROME,ROME v461,official_fr_csv,"93 M18 metiers, 2052 skills, 5146 relations"
6,ONET,O*NET 28.x,official_en_csv,"32 occ, 1506 skills, 6831 relations"
7,SKILL_ONTO,-,neutral_skill_ontology,"5474 skills classified, 131 soft, 13 sub-domains, 5487 edges"
8,MERGE,-,connected_components(exactMatch),"233 unified occ (13 merged), 5405 unified skills (64 merged)"


## 1. Integrity & coverage
Runs the pipeline's own QA (read-only): dangling edges, occupation orphans, unclassified
skills, IT-scope leakage, low-confidence attachments. All should be 0 except the review flag.

In [2]:
from src import pipeline
_ = pipeline.qa()


=== QA ===
occupations: 273 (250 real, 23 ISCO groups)
skills: 5474 (+15 taxonomy nodes)  |  hierarchy edges: 5757  |  dangling: 0
EN label coverage (real occ): 157/250
occupation orphans (no hierarchy parent): 0
low-confidence ISCO attachments (review): 23
skills not placed in ontology: 0
non-IT ISCO group leakage: 0


In [3]:
print('Occupations by source:'); print(occ['source'].value_counts())
print('\nSkills by source (TAXONOMY = ontology nodes):'); print(skl['source'].value_counts())
real = occ[occ.occupation_type != 'isco_group']
print('\nEN label coverage (real occ):', (real.pref_label_en != '').sum(), '/', len(real))
print('label_language_status:'); print(real.label_language_status.value_counts())

Occupations by source:
source
ESCO    110
ROME     93
ONET     32
ISCO     23
NOC      15
Name: count, dtype: int64

Skills by source (TAXONOMY = ontology nodes):
source
ROME        2052
ESCO        1916
ONET        1506
TAXONOMY      15
Name: count, dtype: int64

EN label coverage (real occ): 157 / 250
label_language_status:
label_language_status
en_plus_fr    125
fr_only        93
en_native      32
Name: count, dtype: int64


## 2. De-duplication — unified concepts
Multi-source unified concepts are the actual cross-source integration (source-neutral labels).

In [4]:
uocc['n'] = n_members(uocc.member_entity_ids)
merged = uocc[uocc.n > 1].sort_values('n', ascending=False)
print(f'{len(merged)} multi-source unified occupations of {len(uocc)} total')
merged[['primary_label_en', 'primary_label_fr', 'isco_code', 'sources', 'n']].head(25)

13 multi-source unified occupations of 233 total


,primary_label_en,primary_label_fr,isco_code,sources,n
20,Data Scientists,Data scientist,2511,ESCO | NOC | ONET | ROME,4
167,Computer Systems Engineers/Architects,Ingénieur / Ingénieure Cybersécurité Datacenter,3513,ONET | ROME,4
16,data engineer,Data engineer,2511,ESCO | ROME,2
32,Computer User Support Specialists,technicien informatique/technicienne informatique,3512,ESCO | ONET,2
24,cloud architect,Architecte cloud,2512,ESCO | ROME,2
66,database administrator,administrateur de base de données/administratrice de base de données,2521,ESCO | ONET,2
88,web developer,développeur web/développeuse web,2513,ESCO | ONET,2
101,chief data officer,Chief Data Officer,1330,ESCO | ROME,2
95,data analyst,analyste de données,2511,ESCO | ROME,2
105,software developer,développeur de logiciels/développeuse de logiciels,2512,ESCO | ONET,2


In [5]:
uskl['n'] = n_members(uskl.member_entity_ids)
ms = uskl[uskl.n > 1].sort_values('n', ascending=False)
print(f'{len(ms)} multi-source unified skills of {len(uskl)} total')
ms[['primary_label_en', 'primary_label_fr', 'hard_soft', 'it_subtype', 'sources', 'n']].head(25)

64 multi-source unified skills of 5405 total


,primary_label_en,primary_label_fr,hard_soft,it_subtype,sources,n
716,Database management systems,Systèmes de gestion de base de données,hard,data_databases,ESCO | ONET | ROME,4
559,Microsoft Visual Basic,Visual Basic,hard,other_hard,ESCO | ONET | ROME,3
1137,MySQL,MySQL,hard,other_hard,ESCO | ONET | ROME,3
829,Ruby,Ruby,hard,programming_languages,ESCO | ONET | ROME,3
211,IBM Informix,IBM Informix,hard,other_hard,ESCO | ONET,2
249,manage human resources,Gérer les ressources humaines,hard,other_hard,ESCO | ROME,2
15,Joomla,Joomla,hard,other_hard,ESCO | ONET,2
97,Mathematics,mathématiques,hard,other_hard,ESCO | ONET,2
299,Microsoft Visio,Microsoft Visio,hard,other_hard,ESCO | ONET,2
319,Oracle Warehouse Builder,Oracle Warehouse Builder,hard,data_databases,ESCO | ONET,2


## 3. Alignment & the NLI merge gate
`relation` is the SKOS link; the `merge` flag is what de-duplication consumes — `label`
(identical preferred label) or `semantic` (strong embedding + mutual NLI entailment, same ISCO).

In [6]:
print('SKOS relations:'); print(align.relation.value_counts())
print('\nmerge flag (label/semantic drive de-duplication):'); print(align['merge'].value_counts())
sem = align[align['merge'] == 'semantic']
print(f'\n{len(sem)} semantic (NLI-verified) merges — sample:')
sem[['source_a', 'source_b', 'confidence', 'method', 'notes']].head(15)

SKOS relations:
relation
skos:relatedMatch    44679
skos:closeMatch       2080
skos:exactMatch         84
Name: count, dtype: int64

merge flag (label/semantic drive de-duplication):
merge
            46737
label          84
semantic       22
Name: count, dtype: int64

22 semantic (NLI-verified) merges — sample:


,source_a,source_b,confidence,method,notes
528,ESCO,NOC,0.7827,embed:0.77+nli:0.61,software developer <> Web developers and programmers
552,ESCO,ONET,0.8517,embed:0.75+nli:0.84,ICT security administrator <> Information Security Engineers
712,ESCO,ONET,0.8102,embed:0.74+nli:0.70,ICT technician <> Information Security Engineers
714,ESCO,ONET,0.8191,embed:0.73+nli:0.73,ICT technician <> Computer User Support Specialists
754,ESCO,ONET,0.8119,embed:0.72+nli:0.71,embedded systems software developer <> Web Developers
1072,ESCO,ONET,0.8432,embed:0.76+nli:0.81,ICT operations manager <> Computer and Information Systems Managers
1078,ESCO,ONET,0.8527,embed:0.72+nli:0.84,software developer <> Web Developers
1096,ESCO,ONET,0.8906,embed:0.72+nli:0.97,broadcast technician <> Telecommunications Engineering Specialists
1294,ESCO,ROME,0.8166,embed:0.74+nli:0.72,ICT technician <> Ingénieur informaticien / Ingénieure informaticienne
1503,ESCO,ROME,0.8993,embed:0.78+nli:1.00,ICT security technician <> Technicien / Technicienne en cybersécurité


## 4. ISCO attachment (backbone)
Every ONET/NOC/ROME occupation attaches directly to an ISCO group (NLI-re-ranked). Weak
placements are flagged `ATTACH_LOWCONF` for review — never dropped (0 orphans).

In [7]:
att = hier[hier.source.isin(['ATTACH', 'ATTACH_LOWCONF'])].copy()
print('attach edges:'); print(att.source.value_counts())
occ_lbl = occ.set_index('entity_id').pref_label_en
occ_fr  = occ.set_index('entity_id').pref_label_fr
grp_code = occ.set_index('entity_id').source_code
low = att[att.source == 'ATTACH_LOWCONF'].copy()
low['occupation'] = low.child_entity_id.map(occ_lbl).where(lambda s: s != '', low.child_entity_id.map(occ_fr))
low['isco_group'] = low.parent_entity_id.map(grp_code)
print(f'\n{len(low)} low-confidence attachments (review):')
low[['occupation', 'isco_group']]

attach edges:
source
ATTACH            117
ATTACH_LOWCONF     23
Name: count, dtype: int64

23 low-confidence attachments (review):


,occupation,isco_group
117,"Computer Occupations, All Other",3511
118,Data scientist,2523
119,Chief Data Officer,2521
120,Intégrateur / Intégratrice logiciels métiers,2512
121,Spécialiste Jumeau Numérique,2523
122,Homologateur / Homologatrice fonctionnel de logiciel,2512
123,Architecte systèmes et réseaux des territoires connectés,3513
124,Directeur / Directrice de projets des territoires connectés,1330
125,Architecte IoT - Internet des Objets,2511
126,Ingénieur / Ingénieure Cybersécurité Datacenter,3513


## 5. Skill hierarchy — neutral ontology
Every skill of every source is placed as `skill → sub-domain → Hard/Soft` (no source shapes
the tree). Type/sub-domain nodes are stored as `TAXONOMY` rows; edges are tagged `SKILL_ONTO`.

In [8]:
real_skl = skl[~skl.esco_skill_type.isin(['skill_type', 'skill_domain'])]
print('hard/soft:'); print(real_skl.hard_soft_provisional.value_counts())
print('\nskills per IT sub-domain:'); print(real_skl.it_subtype.value_counts())
taxo = skl[skl.esco_skill_type.isin(['skill_type', 'skill_domain'])]
print('\nontology nodes (types + sub-domains):')
taxo[['pref_label_en', 'esco_skill_type', 'hard_soft_provisional']]

hard/soft:
hard_soft_provisional
hard    5343
soft     131
Name: count, dtype: int64

skills per IT sub-domain:
it_subtype
other_hard                2822
programming_languages      744
knowledge_general          447
security                   323
data_databases             308
networks                   239
soft_transversal           131
web                        116
it_management              109
systems_infrastructure     106
cloud_devops                74
ai_ml                       31
methodology                 24
Name: count, dtype: int64

ontology nodes (types + sub-domains):


,pref_label_en,esco_skill_type,hard_soft_provisional
5474,Hard skill,skill_type,hard
5475,Soft skill,skill_type,soft
5476,Security & cybersecurity,skill_domain,hard
5477,AI & machine learning,skill_domain,hard
5478,Data & databases,skill_domain,hard
5479,Cloud & DevOps,skill_domain,hard
5480,Web development,skill_domain,hard
5481,Networks & telecom,skill_domain,hard
5482,Programming & software development,skill_domain,hard
5483,Systems & infrastructure,skill_domain,hard


In [9]:
# hierarchy edge types present in kb/hierarchy.csv
print('hierarchy edges by source:'); print(hier.source.value_counts())
flat = real_skl[~real_skl.entity_id.isin(hier[hier.entity_kind == 'skill'].child_entity_id)]
orph = occ[(occ.occupation_type != 'isco_group') &
           (~occ.entity_id.isin(hier[hier.entity_kind == 'occupation'].child_entity_id))]
print(f'\nflat skills (no ontology parent): {len(flat)}   |   occupation orphans: {len(orph)}')

hierarchy edges by source:
source
SKILL_ONTO        5487
ATTACH             117
ESCO               110
ATTACH_LOWCONF      23
ISCO                20
Name: count, dtype: int64

flat skills (no ontology parent): 0   |   occupation orphans: 0


## 6. Sources registry
Registered sources and their flags. Add one with `python run_pipeline.py --add NAME`
(subclass `StructuredSource` in `src/sources/`); remove with `--remove NAME`.

In [10]:
from src.sources import registry
pd.DataFrame([
    {'source': n, 'builtin': s.builtin,
     'contributes_occupations': s.contributes_occupations, 'needs_attach': s.needs_attach}
    for n, s in registry.REGISTRY.items()
])

,source,builtin,contributes_occupations,needs_attach
0,ISCO,True,False,False
1,ESCO,True,True,False
2,ONET,True,True,True
3,NOC,True,True,True
4,ROME,True,True,True
5,DEMO,False,True,True
